# Notebook 07 — Rebuild HealthChat with Real Utterances

After running `02b_download_health11k_text.ipynb`, this notebook:
1. Joins HealthChat metadata with the downloaded conversation text
2. Patches the structural parquet to replace `None` utterances with real text
3. Regenerates semantic, conversational, and final outputs

**Run order:** 02b → 07 → done. Notebooks 03–06 do not need to be re-run manually;
this notebook contains a self-contained rebuild pipeline for HealthChat only.

In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
tqdm.pandas()

RAW_DIR      = Path('../data/raw')
TEXT_DIR     = RAW_DIR / 'health11k_text'
STRUCT_DIR   = Path('../data/processed/structural')
SEMANTIC_DIR = Path('../data/processed/semantic')
CONV_DIR     = Path('../data/processed/conversational')
FINAL_DIR    = Path('../data/processed/final')

# Check what's available
wc_path   = TEXT_DIR / 'wildchat_conversations.parquet'
lmsys_path = TEXT_DIR / 'lmsys_conversations.parquet'

has_wildchat = wc_path.exists()
has_lmsys    = lmsys_path.exists()

print('WildChat text available:', has_wildchat)
print('LMSYS text available:   ', has_lmsys)

if not has_wildchat and not has_lmsys:
    raise FileNotFoundError('No text data found. Run 02b_download_health11k_text.ipynb first.')

## 1. Build conversation-level text lookup

In [ ]:
# Load and combine available text
frames = []
if has_wildchat:
    wc = pd.read_parquet(wc_path)
    wc['source'] = 'wildchat'
    frames.append(wc)
    print(f'WildChat: {wc["conversation_id"].nunique():,} conversations, {len(wc):,} turns')

if has_lmsys:
    lm = pd.read_parquet(lmsys_path)
    lm['source'] = 'lmsys'
    frames.append(lm)
    print(f'LMSYS: {lm["conversation_id"].nunique():,} conversations, {len(lm):,} turns')

text_df = pd.concat(frames, ignore_index=True)
print(f'\nTotal text turns: {len(text_df):,}')
print('Columns:', text_df.columns.tolist())
print()
display(text_df.head(4))

In [ ]:
# Normalise role names to match our speaker convention
ROLE_MAP = {
    'user':      'user',
    'human':     'user',
    'assistant': 'assistant',
    'gpt':       'assistant',
    'model':     'assistant',
    'chatgpt':   'assistant',
}

text_df['speaker'] = text_df['role'].str.lower().map(ROLE_MAP).fillna('other')

# Build lookup: (conversation_id, turn_index) -> content
text_lookup = text_df.set_index(['conversation_id', 'turn_index'])['content'].to_dict()
print(f'Lookup size: {len(text_lookup):,} entries')

# Sample
sample_key = list(text_lookup.keys())[0]
print('Sample key:', sample_key)
print('Sample text:', str(text_lookup[sample_key])[:200])

## 2. Patch the structural parquet

In [ ]:
struct_df = pd.read_parquet(STRUCT_DIR / 'harmonized_structural.parquet')
print('Structural dataset:', struct_df.shape)

hc_mask = struct_df['source_dataset'] == 'HealthChat'
print(f'HealthChat rows: {hc_mask.sum():,}')
print(f'HealthChat rows with None utterance: {struct_df[hc_mask]["utterance"].isna().sum():,}')

In [ ]:
def fill_utterance(row):
    """Look up utterance text for a HealthChat row using conversation_id + turn_id."""
    if row['source_dataset'] != 'HealthChat':
        return row['utterance']
    key = (row['original_id'], row['turn_id'])
    text = text_lookup.get(key)
    return text  # None if not found

print('Patching utterances for HealthChat rows...')
struct_df['utterance'] = struct_df.progress_apply(fill_utterance, axis=1)

hc_filled = struct_df[hc_mask]['utterance'].notna().sum()
hc_total  = hc_mask.sum()
print(f'\nFilled: {hc_filled:,} / {hc_total:,} HealthChat turns ({100*hc_filled/hc_total:.1f}%)')

# Spot check
filled_samples = struct_df[hc_mask & struct_df['utterance'].notna()][['dialogue_id','turn_id','speaker','utterance']].head(4)
display(filled_samples)

In [ ]:
# Overwrite the structural parquet with filled utterances
struct_df.to_parquet(STRUCT_DIR / 'harmonized_structural.parquet', index=False)
struct_df.to_csv(STRUCT_DIR / 'harmonized_structural.csv', index=False)
print('Saved patched structural parquet and CSV.')

## 3. Re-run Semantic Harmonization on HealthChat rows

In [ ]:
# ── Normalization map (copy from notebook 04) ────────────────────────────
FOCUS_NORMALIZATION = {
    'diabetes': 'diabetes', 'diabetes mellitus': 'diabetes', 'diabetic': 'diabetes',
    'type 1 diabetes': 'diabetes_type1', 'type 2 diabetes': 'diabetes_type2',
    'heart attack': 'myocardial_infarction', 'myocardial infarction': 'myocardial_infarction',
    'heart disease': 'heart_disease', 'cardiovascular disease': 'heart_disease',
    'hypertension': 'hypertension', 'high blood pressure': 'hypertension',
    'depression': 'depression', 'major depressive disorder': 'depression',
    'anxiety': 'anxiety', 'anxiety disorder': 'anxiety', 'mental health': 'mental_health',
    'cancer': 'cancer', 'tumor': 'cancer', 'neoplasm': 'cancer',
    'asthma': 'asthma', 'copd': 'copd', 'chronic obstructive pulmonary disease': 'copd',
    'migraine': 'migraine', 'headache': 'headache',
    'stroke': 'stroke', 'arthritis': 'arthritis',
    'covid': 'covid_19', 'covid-19': 'covid_19', 'coronavirus': 'covid_19',
    'influenza': 'influenza', 'flu': 'influenza',
    'ibs': 'ibs', 'irritable bowel syndrome': 'ibs',
    'gerd': 'gerd', 'acid reflux': 'gerd',
    'thyroid': 'thyroid_disorder',
    'glaucoma': 'glaucoma',
}

def normalize_focus(focus_raw):
    if focus_raw is None or (isinstance(focus_raw, float) and np.isnan(focus_raw)):
        return None
    key = str(focus_raw).lower().strip()
    if key in FOCUS_NORMALIZATION:
        return FOCUS_NORMALIZATION[key]
    for k, v in FOCUS_NORMALIZATION.items():
        if k in key:
            return v
    cleaned = re.sub(r'[^a-z0-9]+', '_', key).strip('_')
    return cleaned if cleaned else None

# ── Intent patterns (copy from notebook 04) ─────────────────────────────
INTENT_PATTERNS_USER = [
    ('treatment_inquiry',   re.compile(r'\b(treat|treatment|therapy|cure|manage|medication for|medicine for|drug for)\b', re.I)),
    ('medication_inquiry',  re.compile(r'\b(medication|medicine|drug|drugs|pill|prescription|dose|side effect)\b', re.I)),
    ('symptom_inquiry',     re.compile(r'\b(symptom|sign|feel|feeling|experience|suffer|complaint)\b', re.I)),
    ('diagnosis_inquiry',   re.compile(r'\b(diagnos|test|screening|detect|identify)\b', re.I)),
    ('test_or_diagnosis',   re.compile(r'\b(test|exam|lab|imaging|scan|x-ray|mri|ct scan|biopsy)\b', re.I)),
    ('prevention',          re.compile(r'\b(prevent|prevention|avoid|reducing risk|lower risk)\b', re.I)),
    ('risk_factors',        re.compile(r'\b(risk factor|risk of|who is at risk|prone to)\b', re.I)),
    ('cause_or_mechanism',  re.compile(r'\b(cause|causes|why|mechanism|what leads to|how does|origin|etiology)\b', re.I)),
    ('prognosis',           re.compile(r'\b(prognos|outcome|survival|life expectancy|recover|how long)\b', re.I)),
    ('emergency_or_urgent', re.compile(r'\b(emergency|urgent|immediately|chest pain|can.t breathe|severe|911)\b', re.I)),
    ('information_seeking', re.compile(r'\b(what is|what are|tell me|explain|describe|definition|overview|information about)\b', re.I)),
]

FOCUS_KEYWORD_PATTERNS = [
    ('diabetes',             re.compile(r'\bdiabet\w*\b', re.I)),
    ('hypertension',         re.compile(r'\b(hypertension|high blood pressure)\b', re.I)),
    ('myocardial_infarction',re.compile(r'\b(heart attack|myocardial infarction)\b', re.I)),
    ('heart_disease',        re.compile(r'\b(heart disease|cardiovascular|cardiac)\b', re.I)),
    ('depression',           re.compile(r'\b(depress\w+|major depressive)\b', re.I)),
    ('anxiety',              re.compile(r'\banxiet\w+\b', re.I)),
    ('asthma',               re.compile(r'\basthma\b', re.I)),
    ('cancer',               re.compile(r'\b(cancer|tumor|tumour|neoplasm|carcinoma)\b', re.I)),
    ('back_pain',            re.compile(r'\b(back pain|lumbar|spinal)\b', re.I)),
    ('migraine',             re.compile(r'\b(migraine)\b', re.I)),
    ('headache',             re.compile(r'\bheadache\b', re.I)),
    ('stroke',               re.compile(r'\bstroke\b', re.I)),
    ('covid_19',             re.compile(r'\b(covid|coronavirus)\b', re.I)),
    ('influenza',            re.compile(r'\b(influenza|\bflu\b)\b', re.I)),
    ('pain',                 re.compile(r'\b(pain|ache|sore)\b', re.I)),
    ('fever',                re.compile(r'\bfever\b', re.I)),
]

MEDICAL_ENTITIES = [
    ('diabetes', re.compile(r'\bdiabet\w*\b', re.I)),
    ('hypertension', re.compile(r'\b(hypertension|high blood pressure)\b', re.I)),
    ('cancer', re.compile(r'\b(cancer|carcinoma|neoplasm|tumor)\b', re.I)),
    ('depression', re.compile(r'\b(depression|depressive)\b', re.I)),
    ('anxiety', re.compile(r'\banxiet\w+\b', re.I)),
    ('asthma', re.compile(r'\basthma\b', re.I)),
    ('stroke', re.compile(r'\bstroke\b', re.I)),
    ('chest pain', re.compile(r'\bchest pain\b', re.I)),
    ('nausea', re.compile(r'\bnausea\b', re.I)),
    ('fever', re.compile(r'\bfever\b', re.I)),
    ('fatigue', re.compile(r'\b(fatigue|tired|exhaustion)\b', re.I)),
    ('headache', re.compile(r'\bheadache\b', re.I)),
    ('back pain', re.compile(r'\bback pain\b', re.I)),
    ('insulin', re.compile(r'\binsulin\b', re.I)),
    ('metformin', re.compile(r'\bmetformin\b', re.I)),
    ('antibiotics', re.compile(r'\bantibiotic\w*\b', re.I)),
    ('blood glucose', re.compile(r'\b(blood glucose|blood sugar|hba1c|a1c)\b', re.I)),
    ('blood pressure', re.compile(r'\bblood pressure\b', re.I)),
    ('cholesterol', re.compile(r'\bcholesterol\b', re.I)),
    ('surgery', re.compile(r'\b(surgery|surgical|operation)\b', re.I)),
    ('mri', re.compile(r'\bmri\b', re.I)),
    ('covid-19', re.compile(r'\b(covid.?19|coronavirus)\b', re.I)),
]

TAXONOMY_INTENT_MAP = {
    'A1.1': 'symptom_inquiry', 'A1.2': 'symptom_inquiry',
    'A1.3': 'diagnosis_inquiry', 'A1.5': 'information_seeking',
    'B1': 'information_seeking', 'B2': 'treatment_inquiry',
    'B3.3': 'medication_inquiry', 'B4': 'prevention',
    'B5.1': 'cause_or_mechanism', 'B5.2': 'risk_factors',
    'B7': 'prognosis', 'B8': 'symptom_inquiry',
    'B9': 'information_seeking', 'B10': 'treatment_inquiry',
    'C5': 'follow_up', 'C6': 'emergency_or_urgent',
    'D1': 'information_seeking',
}

def infer_focus_from_text(text):
    if not text or (isinstance(text, float) and np.isnan(text)):
        return None, 0.0
    for label, pattern in FOCUS_KEYWORD_PATTERNS:
        if pattern.search(str(text)):
            return label, 0.6
    return None, 0.0

def infer_intent_from_text(text, speaker='user'):
    if speaker == 'assistant':
        return 'other', 0.9
    if not text or (isinstance(text, float) and np.isnan(text)):
        return 'other', 0.3
    for intent, pattern in INTENT_PATTERNS_USER:
        if pattern.search(str(text)):
            return intent, 0.7
    return 'information_seeking', 0.5

def extract_entities(text):
    if not text or (isinstance(text, float) and np.isnan(text)):
        return []
    found, seen = [], set()
    for entity, pattern in MEDICAL_ENTITIES:
        if entity not in seen and pattern.search(str(text)):
            found.append(entity)
            seen.add(entity)
    return found

print('Helper functions loaded.')

In [ ]:
# Load the existing semantic parquet — patch only HealthChat rows where utterance is now filled
sem_df = pd.read_parquet(SEMANTIC_DIR / 'harmonized_semantic.parquet')
print('Existing semantic dataset:', sem_df.shape)

# Get the newly patched structural data for HealthChat
hc_struct = struct_df[struct_df['source_dataset'] == 'HealthChat'].copy()
hc_filled_mask = hc_struct['utterance'].notna()
print(f'HealthChat rows with text now available: {hc_filled_mask.sum():,}')

# Re-annotate focus, intent, entities for filled rows
hc_filled = hc_struct[hc_filled_mask].copy()

focus_raw_list, focus_norm_list, intent_list, entity_list = [], [], [], []
intent_conf_list, focus_conf_list = [], []

for _, row in tqdm(hc_filled.iterrows(), total=len(hc_filled), desc='Re-annotate HealthChat'):
    text = row['utterance']
    speaker = row['speaker']
    label_raw = row['source_label_raw']

    # Focus: try specialty from metadata first, then infer from text
    focus_raw, focus_norm, fconf = None, None, 0.0
    try:
        meta = json.loads(str(label_raw))
        specialty = meta.get('specialty')
        if specialty and specialty != 'Other':
            focus_raw = specialty
            focus_norm = normalize_focus(specialty)
            fconf = 0.9
        else:
            # Now we have text — try inference
            focus_raw, fconf = infer_focus_from_text(text)
            focus_norm = normalize_focus(focus_raw) if focus_raw else None
    except Exception:
        focus_raw, fconf = infer_focus_from_text(text)
        focus_norm = normalize_focus(focus_raw) if focus_raw else None

    # Intent
    intent, iconf = infer_intent_from_text(text, speaker)

    # Entities
    entities = extract_entities(text)

    focus_raw_list.append(focus_raw)
    focus_norm_list.append(focus_norm)
    focus_conf_list.append(fconf)
    intent_list.append(intent)
    intent_conf_list.append(iconf)
    entity_list.append(entities)

hc_filled = hc_filled.assign(
    focus_raw=focus_raw_list,
    focus_normalized=focus_norm_list,
    primary_intent=intent_list,
    medical_entities=entity_list,
)

print('Re-annotation complete.')

In [ ]:
# Merge updated HealthChat rows back into semantic dataset
# Strategy: replace HealthChat rows in sem_df with re-annotated versions

non_hc_sem = sem_df[sem_df['source_dataset'] != 'HealthChat'].copy()

# The sem_df HealthChat rows need to be rebuilt from struct + new annotations
# Use the structural baseline for rows without text (lmsys if not downloaded yet)
hc_all_struct = struct_df[struct_df['source_dataset'] == 'HealthChat'].copy()

# For rows without text, carry over existing semantic annotations
hc_no_text = hc_all_struct[hc_all_struct['utterance'].isna()].copy()
if len(hc_no_text) > 0:
    # Merge with existing semantic columns
    existing_hc_sem = sem_df[sem_df['source_dataset'] == 'HealthChat'].copy()
    existing_hc_sem = existing_hc_sem.set_index(['dialogue_id', 'turn_id'])

    for col in ['focus_raw', 'focus_normalized', 'primary_intent', 'dialogue_act',
                'medical_entities', 'annotation_source', 'annotation_confidence']:
        if col in existing_hc_sem.columns:
            idx = list(zip(hc_no_text['dialogue_id'], hc_no_text['turn_id']))
            hc_no_text[col] = [existing_hc_sem.loc[i, col] if i in existing_hc_sem.index else None for i in idx]

# Add dialogue_act inference for filled rows
RE_QUESTION = re.compile(r'\?\s*$')
RE_QUESTION_WORD = re.compile(r'^\s*(what|how|why|when|where|who|which|can|could|should|is|are|do|does|did|has|will|would)\b', re.I)
RE_GREETING = re.compile(r'^\s*(hello|hi|hey|good morning|dear doctor)\b', re.I)

def quick_dialogue_act(speaker, text):
    if not text or (isinstance(text, float) and np.isnan(text)):
        return ('question' if speaker == 'user' else 'answer')
    t = str(text).strip()
    words = t.split()
    if RE_GREETING.match(t) and len(words) <= 10:
        return 'greeting'
    if speaker == 'user':
        if RE_QUESTION.search(t) or RE_QUESTION_WORD.match(t):
            return 'question'
        return 'statement'
    if RE_QUESTION.search(t) and len(words) < 30:
        return 'clarification_request'
    return 'answer'

hc_filled['dialogue_act'] = hc_filled.apply(
    lambda r: quick_dialogue_act(r['speaker'], r['utterance']), axis=1
)
hc_filled['annotation_source'] = 'rule_based'
hc_filled['annotation_confidence'] = hc_filled.apply(
    lambda r: round((focus_conf_list[0] if r.name == hc_filled.index[0] else 0.7), 3), axis=1
)
# Simpler: set confidence uniformly for re-annotated rows
conf_vals = []
for fr, ic in zip(focus_conf_list, intent_conf_list):
    vals = [v for v in [fr, ic] if v > 0]
    conf_vals.append(round(sum(vals)/len(vals), 3) if vals else 0.6)
hc_filled['annotation_confidence'] = conf_vals

# Ensure all required semantic columns exist
for col in non_hc_sem.columns:
    if col not in hc_filled.columns:
        hc_filled[col] = None
    if col not in hc_no_text.columns:
        hc_no_text[col] = None

# Combine
new_sem = pd.concat([non_hc_sem, hc_filled, hc_no_text], ignore_index=True)
new_sem = new_sem[sem_df.columns]  # keep same column order
new_sem = new_sem.sort_values(['source_dataset', 'dialogue_id', 'turn_id']).reset_index(drop=True)

new_sem.to_parquet(SEMANTIC_DIR / 'harmonized_semantic.parquet', index=False)
print('Saved updated semantic parquet.')
print('Shape:', new_sem.shape)
hc_text_now = new_sem[(new_sem['source_dataset']=='HealthChat') & new_sem['utterance'].notna()]
print(f'HealthChat turns with utterance: {len(hc_text_now):,}')

## 4. Re-run Conversational Harmonization

In [ ]:
# Re-run notebook 05 logic on the updated semantic dataset
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook',
     '--execute', '--inplace', '--ExecutePreprocessor.timeout=600',
     'notebooks/05_harmonize_conversation.ipynb'],
    capture_output=True, text=True
)
print('stdout:', result.stdout[-500:] if result.stdout else '')
print('stderr (last 500):', result.stderr[-500:] if result.stderr else '')
if result.returncode not in (0, 1):  # 1 = warnings only
    print('WARNING: notebook may have failed, check output')
else:
    print('Notebook 05 completed.')

## 5. Re-run Validation and Export

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook',
     '--execute', '--inplace', '--ExecutePreprocessor.timeout=600',
     'notebooks/06_validate_export.ipynb'],
    capture_output=True, text=True
)
print('stderr (last 500):', result.stderr[-500:] if result.stderr else '')
if result.returncode not in (0, 1):
    print('WARNING: notebook may have failed')
else:
    print('Notebook 06 completed.')

In [ ]:
# Final summary
with open(FINAL_DIR / 'dataset_statistics.json') as f:
    stats = json.load(f)

print('=== UPDATED DATASET STATISTICS ===')
print('Total dialogues:', stats['total_dialogues'])
print('Total turns:', stats['total_turns'])
print('Dialogues per source:', stats['dialogues_per_source'])
print('Turns per source:', stats['turns_per_source'])
print()

final = pd.read_parquet(FINAL_DIR / 'harmonized_healthcare_dataset.parquet')
hc_with_text = final[(final['source_dataset']=='HealthChat') & final['utterance'].notna()]
hc_without = final[(final['source_dataset']=='HealthChat') & final['utterance'].isna()]
print('HealthChat turns WITH text:', len(hc_with_text))
print('HealthChat turns WITHOUT text (lmsys, pending):', len(hc_without))